# 🍣 AlexNet COMBINED - Tim Member 4

Notebook ini untuk melatih **AlexNet Combined** (BatchNorm + Dropout + improvements).

**Assigned to:** [NAMA MEMBER 4]

---

In [6]:
# ============================================================
# SETUP - Jalankan cell ini terlebih dahulu!
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_PATH = '/content/drive/MyDrive/AlexNet_iFood2019'
REPO_PATH = '/content/alexnet-ifood2019'

# Clone repo
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/deftorch/alexnet-ifood2019.git {REPO_PATH}

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

# Install dependencies
!pip install -q torch torchvision pandas numpy pillow scikit-learn matplotlib seaborn tqdm wandb

# Create symlinks to Drive
!rm -rf data checkpoints evaluation_results analysis_results 2>/dev/null
!ln -s {PROJECT_PATH}/dataset data
!ln -s {PROJECT_PATH}/checkpoints checkpoints
!ln -s {PROJECT_PATH}/evaluation_results evaluation_results
!ln -s {PROJECT_PATH}/analysis_results analysis_results

import torch
print(f"\n✅ Setup Complete!")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Setup Complete!
PyTorch: 2.9.0+cu126
GPU: Tesla T4


In [8]:
# ============================================================
# TRAINING: AlexNet Combined
# ============================================================

MODEL_NAME = "alexnet_combined"

# Konfigurasi training
NUM_EPOCHS = 50
BATCH_SIZE = 128
LR = 0.01

print(f"🚀 Training {MODEL_NAME}...")
print(f"   Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}")

!python src/train.py \
    --data_dir data \
    --model_name {MODEL_NAME} \
    --num_epochs {NUM_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --save_dir checkpoints

🚀 Training alexnet_combined...
   Epochs: 50, Batch: 128, LR: 0.01
Using device: cuda
GPU: Tesla T4

Loading data...
Traceback (most recent call last):
  File "/content/alexnet-ifood2019/src/train.py", line 425, in <module>
    main()
  File "/content/alexnet-ifood2019/src/train.py", line 290, in main
    raise ValueError("Train and val dataloaders required")
ValueError: Train and val dataloaders required


In [4]:
# ============================================================
# EVALUATION
# ============================================================

MODEL_NAME = "alexnet_combined"

print(f"📊 Evaluating {MODEL_NAME}...")

!python src/evaluate.py \
    --data_dir data \
    --model_path checkpoints/{MODEL_NAME}_best.pth \
    --model_name {MODEL_NAME} \
    --split val \
    --output_dir evaluation_results

print(f"\n✅ Evaluation selesai! Hasil tersimpan di: evaluation_results/{MODEL_NAME}_val_metrics.json")

📊 Evaluating alexnet_combined...
Using device: cuda

Loading model: alexnet_combined
Checkpoint: checkpoints/alexnet_combined_best.pth
Traceback (most recent call last):
  File "/content/alexnet-ifood2019/src/evaluate.py", line 308, in <module>
    main()
  File "/content/alexnet-ifood2019/src/evaluate.py", line 267, in main
    model = load_model(args.model_path, args.model_name, args.num_classes, device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/alexnet-ifood2019/src/evaluate.py", line 66, in load_model
    checkpoint = torch.load(model_path, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1484, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 759, in _open_file_like
    return _open_file(name_or_b

In [5]:
# ============================================================
# LIHAT HASIL
# ============================================================

import json
import matplotlib.pyplot as plt

MODEL_NAME = "alexnet_combined"

# Load history
with open(f'checkpoints/{MODEL_NAME}_history.json') as f:
    history = json.load(f)

# Load metrics
with open(f'evaluation_results/{MODEL_NAME}_val_metrics.json') as f:
    metrics = json.load(f)

# Print summary
print("=" * 50)
print(f"📈 HASIL {MODEL_NAME.upper()}")
print("=" * 50)
print(f"Best Val Accuracy: {max(history['val_acc']):.4f}")
print(f"Best Epoch: {history['val_acc'].index(max(history['val_acc'])) + 1}")
print(f"Test Accuracy: {metrics['accuracy']:.4f}")
print(f"Top-5 Accuracy: {metrics['top5_accuracy']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print("=" * 50)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'], label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(f'{MODEL_NAME} Training Curves')
plt.tight_layout()
plt.savefig(f'analysis_results/{MODEL_NAME}_curves.png', dpi=150)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/alexnet_combined_history.json'